# TAMER OCR v2.1 — Colab / Kaggle Training Notebook

This notebook is an **execution layer only**. All logic lives in `tamer_ocr/`.

**Pipeline (strict order — no training until ALL data is preprocessed):**
1. Authenticate (HuggingFace + Kaggle)
2. Install dependencies
3. Load codebase
4. Preprocess ALL 4 datasets → verify → push to HF dataset repo
5. Train (auto-resumes from latest checkpoint if interrupted)
6. Push final model to HF

Works on **both** Google Colab and Kaggle Notebooks.

## 1. Detect Environment

In [ ]:
import os

# Detect environment
IS_COLAB = os.path.exists('/content')
IS_KAGGLE = os.path.exists('/kaggle')

if IS_COLAB:
    print('Environment: Google Colab')
elif IS_KAGGLE:
    print('Environment: Kaggle')
else:
    print('Environment: Unknown (local?)')

# Check GPU
!nvidia-smi || echo 'No GPU found'

## 2. Authenticate — HuggingFace & Kaggle

In [ ]:
import getpass
import os

# --- HuggingFace Token ---
hf_token = os.getenv('HF_TOKEN', '')
if not hf_token:
    hf_token = getpass.getpass('Enter your HuggingFace token (https://huggingface.co/settings/tokens): ')
os.environ['HF_TOKEN'] = hf_token

# --- Kaggle Credentials ---
kaggle_username = 'merselfares'  # Hardcoded as requested
kaggle_key = os.getenv('KAGGLE_KEY', '')
if not kaggle_key:
    kaggle_key = getpass.getpass('Enter your Kaggle API Key (https://www.kaggle.com/settings/account): ')
os.environ['KAGGLE_USERNAME'] = kaggle_username
os.environ['KAGGLE_KEY'] = kaggle_key

# Write kaggle.json for the CLI
kaggle_dir = os.path.expanduser('~/.kaggle')
os.makedirs(kaggle_dir, exist_ok=True)
with open(os.path.join(kaggle_dir, 'kaggle.json'), 'w') as f:
    f.write(f'{{"username":"{kaggle_username}","key":"{kaggle_key}"}}')
os.chmod(os.path.join(kaggle_dir, 'kaggle.json'), 0o600)

# Login to HuggingFace
from huggingface_hub import login
login(token=hf_token, add_to_git_credential=True)

# Get HF username for auto repo naming
from huggingface_hub import HfApi
hf_api = HfApi(token=hf_token)
hf_username = hf_api.whoami()['name']
print(f'Authenticated as: {hf_username} (HuggingFace), {kaggle_username} (Kaggle)')

## 3. Install Dependencies

In [ ]:
!pip install -q timm albumentations datasets huggingface_hub requests tqdm psutil opencv-python-headless

## 4. Load Codebase

In [ ]:
import os

# --- Choose your method to get the codebase ---
# Option A: Upload tamer_ocr.zip to the notebook and unzip
# Option B: Clone from GitHub
# Option C: Code is already in working directory

# Option A: Upload ZIP (uncomment)
# !unzip -o tamer_ocr.zip -d .

# Option B: Git clone (replace URL)
!git clone https://github.com/Vtheonly/AI_PROJECT_TAMER_Complete
# %cd tamer-ocr

# Option C: Already present
assert os.path.exists('tamer_ocr'), "tamer_ocr/ not found! Upload or clone the codebase first."
assert os.path.exists('train.py'), "train.py not found!"
print('Codebase loaded successfully!')

## 5. Configure Training

In [ ]:
from tamer_ocr.config import Config

config = Config()

# === PATHS ===
if os.path.exists('/kaggle'):
    config.data_dir = '/kaggle/working/data'
    config.output_dir = '/kaggle/working/outputs'
    config.checkpoint_dir = '/kaggle/working/checkpoints'
    config.log_dir = '/kaggle/working/logs'
else:
    config.data_dir = '/content/data'
    config.output_dir = '/content/outputs'
    config.checkpoint_dir = '/content/checkpoints'
    config.log_dir = '/content/logs'

# === HF REPOS ===
config.hf_token = hf_token
config.hf_repo_id = f"{hf_username}/tamer-ocr-model"
config.hf_dataset_repo_id = f"{hf_username}/tamer-preprocessed"

# === KAGGLE ===
config.kaggle_username = 'merselfares'
config.kaggle_key = kaggle_key

# === MODEL (defaults are correct) ===
# config.encoder_name = 'swin_base_patch4_window7_224'
# config.d_model = 512
# config.num_decoder_layers = 6

# === TRAINING ===
config.batch_size = 8
config.accumulation_steps = 4  # Effective batch = 32
config.num_epochs = 150
config.encoder_lr = 1e-5
config.decoder_lr = 1e-4
config.label_smoothing = 0.1
config.total_training_steps = 50000

# === CHECKPOINTING ===
config.checkpoint_every_epochs = 3  # Save every 3 epochs
config.keep_last_n_checkpoints = 3

# === INFERENCE ===
config.beam_width = 5
config.length_penalty = 0.6

print('Configuration ready!')
print(f'  HF model repo: {config.hf_repo_id}')
print(f'  HF dataset repo: {config.hf_dataset_repo_id}')
print(f'  Checkpoint dir: {config.checkpoint_dir}')
print(f'  Effective batch size: {config.batch_size * config.accumulation_steps}')
print(f'  Checkpoint every: {config.checkpoint_every_epochs} epochs')

## 6. Run Full Pipeline

**This runs the strict pipeline:**
1. Preprocess ALL datasets
2. Verify → push to HF dataset repo
3. Build model
4. Auto-resume from latest checkpoint (if interrupted)
5. Train
6. Final beam search evaluation

If your session crashes, just **re-run all cells** — it will auto-resume from the last checkpoint.

In [ ]:
from tamer_ocr.core.trainer import Trainer

trainer = Trainer(config)
trainer.run()  # All logic inside — auto-resumes if interrupted

## 7. Push Final Model to HuggingFace

In [ ]:
import os
from tamer_ocr.utils.checkpoint import push_checkpoint_to_hf

best_path = os.path.join(config.checkpoint_dir, 'best.pt')
if os.path.exists(best_path):
    push_checkpoint_to_hf(best_path, config, epoch=0, is_best=True)
    print(f'Pushed best.pt to {config.hf_repo_id}')
else:
    print('No best.pt found — training may not have completed.')

## 8. Beam Search Evaluation (Optional)

Run a full beam search eval on the best checkpoint.

In [ ]:
import os
from tamer_ocr.utils.checkpoint import find_latest_checkpoint

best_path = os.path.join(config.checkpoint_dir, 'best.pt')
if not os.path.exists(best_path):
    best_path = find_latest_checkpoint(config.checkpoint_dir)

if best_path:
    trainer = Trainer(config)
    trainer.preprocess_data()
    trainer.create_dataloaders()
    trainer.build_model()
    trainer.resume_from_checkpoint(best_path)
    metrics = trainer.evaluate_with_beam_search(max_samples=500)
    print('\nBeam Search Results:')
    for k, v in metrics.items():
        print(f'  {k}: {v:.4f}')
else:
    print('No checkpoint found for evaluation.')